# Import Statement and Dependencies

In [15]:
%load_ext autoreload
%autoreload 2

import sys
import torch
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from Utils.ModelLoaders import *
from Utils.TrainUtils import get_device
from Models.EnsembleModel import *

from Utils.DataUtils import build_ae_dataloaders

base_path = "checkpoints"
DEVICE = get_device()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load In Datasets and loaders

In [16]:
train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


# Load Autoencoder 16 and 256

In [25]:
ae_16_path = f"{base_path}/Autoencoder/ae_best_L16.pt"
ae_256_path = f"{base_path}/Autoencoder/autoencoder.pt"

autoencoder_16 = load_autoencoder(path=ae_16_path, device=DEVICE)
autoencoder_256 = load_autoencoder(path=ae_256_path, device=DEVICE)

# Load Gated MLP 16 and 256

In [19]:
gated_mlp_16_path = f"{base_path}/GatedMLP_AE16_V4/best.pt"
gated_mlp_256_path = f"{base_path}/GatedMLP_AE256_V1/best.pt"


gated_mlp_16 = load_fraud_mlp(path=gated_mlp_16_path, encoder=autoencoder_16 , device=DEVICE)
gated_mlp_256 = load_fraud_mlp(path=gated_mlp_256_path, encoder=autoencoder_256 , device=DEVICE)


# Load CatBoost

In [20]:
import catboost as cb

catboost_path = f"{base_path}/CatBoost/best_catboost.cbm"

catboost_model = cb.CatBoostClassifier()
catboost_model.load_model(catboost_path)

CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=1, depth=10, eval_metric='AUC', iterations=3000, l2_leaf_reg=10, learning_rate=0.03, loss_function='Logloss', od_wait=100, random_seed=42, random_strength=1, verbose=100)

# Load XGBoost

In [21]:
from xgboost import XGBClassifier

xgboost_path = f"{base_path}/XGBoost/best_xgboost.json"

xgb_model = XGBClassifier()
xgb_model.load_model(xgboost_path)

# Load Base Gated MLP

In [22]:
gated_mlp_base_path = f"{base_path}/GatedMLP_V6/best.pt"
gated_mlp_base = load_fraud_mlp(path=gated_mlp_base_path, encoder=None, device=DEVICE)

# Load Ensemble Model

In [27]:
members = [
    TorchBinaryProbWrapper(gated_mlp_base),
    TorchBinaryProbWrapper(gated_mlp_16),
    TorchBinaryProbWrapper(gated_mlp_256),
    CatBoostAEWrapper(catboost_model, autoencoder_256),
    XGBoostAEWrapper(xgb_model, autoencoder_256)
]

stacker = StackingFraudMLP(
    input_dim=len(members),
    hidden_dims=[32, 16, 8],
    dropout=0.1,
    use_norm=True,
).to(DEVICE)

final_ensemble_path = f"{base_path}/FinalEnsemble/StackedEnsemble_FULL_CAT_XGB_TREE_32_16_8_TRAINVAL.pt"
final_ensemble = load_stacked_ensemble(path=final_ensemble_path,members=members ,device=DEVICE)

# TEST Ensemble on test set

In [28]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from Utils.TrainUtils import threshold_sweep, compute_metrics_from_probs

# collect labels + probs
test_labels = collect_labels(test_loader)

val_probs = collect_stacked_ensemble_probs(
    ensemble=final_ensemble,
    loader=val_loader,
    device=DEVICE,
)

val_labels = collect_labels(val_loader)

val_best_metrics, _ = threshold_sweep(
    probs=val_probs,
    labels=val_labels,
    num_thresholds=300,
    optimize_for="f2",
)

best_threshold = val_best_metrics["threshold"]

test_probs = collect_stacked_ensemble_probs(
    ensemble=final_ensemble,
    loader=test_loader,
    device=DEVICE,
)

test_labels = collect_labels(test_loader)

labels_np = test_labels.cpu().numpy()
probs_np = test_probs.cpu().numpy()

preds_np = (probs_np >= best_threshold).astype(int)

# sklearn metrics
acc  = accuracy_score(labels_np, preds_np)
prec = precision_score(labels_np, preds_np)
rec  = recall_score(labels_np, preds_np)
f1   = f1_score(labels_np, preds_np)

# compute F2 manually
beta = 2
f2 = (1 + beta**2) * (prec * rec) / ((beta**2 * prec) + rec + 1e-12)

print("=== FINAL RESULTS (OPTIMISED THRESHOLD) ===")
print(f"Threshold: {best_threshold:.6f}")
print(f"ACCURACY: {acc}")
print(f"PRECISION: {prec}")
print(f"RECALL: {rec}")
print(f"F1: {f1}")
print(f"F2: {f2}")
print()

print(classification_report(labels_np, preds_np, digits=2))

stacked_opti_metrics = compute_metrics_from_probs(
    probs=test_probs,
    labels=test_labels,
    threshold=best_threshold,
)

print(f"=== Stacked Ensemble Test ({best_threshold} threshold) ===")
for k, v in stacked_opti_metrics.items():
    print(f"{k}: {v}")

=== FINAL RESULTS (OPTIMISED THRESHOLD) ===
Threshold: 0.219210
ACCURACY: 0.9816439191248688
PRECISION: 0.7320415879017014
RECALL: 0.749757986447241
F1: 0.7407938785270206
F2: 0.746146435452591

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     56988
         1.0       0.73      0.75      0.74      2066

    accuracy                           0.98     59054
   macro avg       0.86      0.87      0.87     59054
weighted avg       0.98      0.98      0.98     59054

=== Stacked Ensemble Test (0.21921023726463318 threshold) ===
threshold: 0.21921023726463318
tp: 1549
fp: 567
tn: 56421
fn: 517
precision: 0.7320415878982418
recall: 0.749757986443612
f1: 0.7407938735241927
f2: 0.7461464334204836
accuracy: 0.9816439191247026
